# 01 — Exploratory Data Analysis (EDA)

**Goal:** Understand the MVSA-Single dataset before building any model.

## What we analyse
1. **Class distribution** — are the three emotion classes balanced?
2. **Caption length** — how many words does a typical tweet have?
3. **Top words per class** — which words are most associated with each emotion?

## Dataset: MVSA-Single
- 4,869 image-text pairs from Twitter (after filtering)
- Labels: `positive`, `negative`, `neutral`
- Source: http://mcrlab.net/research/mvsa-sentiment-analysis-on-multi-view-social-data/

## Cell 1 — Setup and data loading

In [ ]:
import os
os.chdir("/Users/yesicarb/Desktop/UIE/3º Curso/2 SEM/PROYECTO/emotion/multimodal_emotion")

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import nltk
nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.corpus import stopwords

df = pd.read_csv("data/processed/labels.csv")
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Cell 2 — Class distribution

We expect a slight class imbalance — Twitter travel posts tend to be
more positive or neutral than negative.

This imbalance motivates the use of **macro-averaged F1** as our primary
metric (it penalises equally across all classes, regardless of size).

In [ ]:
colors = {'positive': '#16a34a', 'neutral': '#6b7280', 'negative': '#dc2626'}

fig, ax = plt.subplots(figsize=(6, 4))
df['label'].value_counts().plot(
    kind='bar', ax=ax,
    color=[colors[l] for l in df['label'].value_counts().index]
)
ax.set_title('Class distribution — MVSA Single')
ax.set_xlabel('Emotion label')
ax.set_ylabel('Number of samples')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.savefig('results/figures/class_distribution.png', dpi=150)
plt.show()

print("\nClass counts:")
print(df['label'].value_counts())
print(f"\nClass imbalance ratio (max/min): "
      f"{df['label'].value_counts().max() / df['label'].value_counts().min():.2f}x")

## Cell 3 — Caption length distribution

Twitter has a 280-character limit, so captions are typically short.
Short texts are harder to classify reliably — this motivates using
BERT, which captures contextual meaning even in short sequences.

In [ ]:
df['caption_len'] = df['text'].apply(lambda x: len(str(x).split()))

fig, ax = plt.subplots(figsize=(7, 4))
for label, color in colors.items():
    subset = df[df['label'] == label]['caption_len']
    ax.hist(subset, bins=30, alpha=0.5, label=label, color=color)
ax.set_title('Caption length by class')
ax.set_xlabel('Number of words')
ax.set_ylabel('Frequency')
ax.legend()
plt.tight_layout()
plt.savefig('results/figures/caption_length.png', dpi=150)
plt.show()

print(f"Mean length:    {df['caption_len'].mean():.1f} words")
print(f"Maximum length: {df['caption_len'].max()} words")
print(f"Minimum length: {df['caption_len'].min()} words")

## Cell 4 — Top 10 words per class

We remove English stopwords (e.g. 'the', 'a', 'is') since they carry
no emotional meaning. The remaining words show which vocabulary is
most discriminative for each class.

**Expected findings:**
- Positive: words like 'love', 'happy', 'beautiful', 'amazing'
- Negative: words like 'bad', 'miss', 'lost', 'worst'
- Neutral: generic travel vocabulary without clear emotional polarity

In [ ]:
stop = set(stopwords.words('english'))

for label in ['positive', 'negative', 'neutral']:
    texts = df[df['label'] == label]['text'].tolist()
    words = [
        w.lower() for t in texts
        for w in str(t).split()
        if w.lower() not in stop and w.isalpha() and len(w) > 2
    ]
    top10 = Counter(words).most_common(10)
    print(f"\nTop 10 words — {label}:")
    for word, count in top10:
        bar = '█' * (count // 5)
        print(f"  {word:<15} {count:>4}  {bar}")